In [1]:
!pip install google-generativeai

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached httplib2-0.31.0-py3-none-any.whl.metadata (2.2 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.3 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 5.7 MB/s  0:00:00
   ---------------------------------------- 0.0/14.6 MB ? eta -:--:--
   ------------------- -------------------- 7.1/14.6 MB 33.6 MB/s eta 0:00:01
   ----------------------------------- ---- 12.8/14.6 MB 32.3 MB/s eta 0:00:01
   ------------------------------------ --- 13.4/14.6 MB 24.0 MB/s eta 0:00:01
   ------------------------------------- -- 13.9/14.6 MB 17.5 MB/s eta 0:00:01
   ----------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.15.0 requires numpy<2.0.0,>=1.23.5, but you have numpy 2.2.6 which is incompatible.
tensorflow-intel 2.15.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.5 which is incompatible.


In [2]:
pip install python-dotenv

In [3]:
from dotenv import load_dotenv
load_dotenv()  # поднимет переменные из .env


True

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()  # грузим GEMINI_API_KEY из .env

genai.configure(api_key=os.environ["GEMINI_API_KEY"])

MODEL_NAME = "gemini-1.5-flash"  # можно поменять на pro, если нужно

def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def ask_gemini_about_doc(doc_text: str, user_question: str) -> str:
    model = genai.GenerativeModel(MODEL_NAME)

    prompt = f"""
Ты — AI-Procure, экспертный AI-аналитик по государственным закупкам и тендерной документации.
Твоя роль — автоматически анализировать закупочные документы, искать риски, выявлять признаки 
антиконкурентности, аффилированности и «заточенных» условий, а также генерировать структурированные отчёты.

Работай строго по следующим правилам:

1) Если предоставлен полный текст тендера (PDF, DOCX → распарсенный текст), анализируй ТОЛЬКО его.
   Не придумывай данных, которые в документе отсутствуют.

2) Всегда извлекай ключевые поля:
   - Название закупки
   - Заказчик
   - Бюджет
   - Сроки подачи заявок
   - Предмет закупки
   - Критерии оценки
   - Технические требования
   - Требования к квалификации
   - Условия договора
   - Предыдущие поставщики (если указано)
   - Любые доступные даты, суммы, параметры

3) Формируй раздел "Общее резюме":
   - Краткое содержание закупки (3–5 предложений)
   - Ключевые параметры
   - Что требуется поставщику
   - Особенности предмета закупки

4) Формируй раздел "Анализ рисков".
   Для каждого риска делай:
   - Флаг риска (risk_flag)
   - Описание в 1–2 предложениях
   - Почему это риск (justification)
   - Насколько это серьёзно (0–100)

   Возможные категории рисков:
   - Уникальные или слишком узкие технические требования
   - Подозрительно короткие сроки
   - Завышенная цена относительно типичных закупок
   - Требование редких моделей оборудования
   - Наличие предыдущего единственного поставщика
   - Аффилированность (если документ явно указывает связи)
   - Ограничение конкуренции странными критериями
   - Непропорциональные квалификационные требования
   - Любые изменения документации перед дедлайном (если указано)
   - Признаки копирования старых тендеров

5) Формируй раздел "Структурированный анализ тендера":
   - Структура документа
   - Сбор всех KPI/показателей
   - Условия договора (пени, гарантии, сроки, штрафы)
   - Техническая спецификация в виде таблицы

6) Опционально (если доступно) — "Сравнение с похожими тендерами":
   - Упомяни 3–5 похожих закупок (используй только те, что явно указаны в данных)
   - Сравни цену, условия, требования

7) Стиль ответа:
   - Чётко
   - Структурировано
   - Как профессиональный аналитик, без "воды"
   - Не используй фразы «как ИИ-модель» и т.п.

8) Если информации недостаточно — честно говори "В документе нет данных о …".
   Не выдумывай.

Формат ответа всегда такой:

=== SUMMARY ===
(краткое резюме)

=== KEY FIELDS ===
- title:
- customer:
- budget:
- deadline:
- … (другие извлеченные поля)

=== RISK ANALYSIS ===
Каждый риск в формате:
- risk_flag:
- severity (0–100):
- explanation:
- justification:

=== TECHNICAL ANALYSIS ===
(структура документа, KPI, табличка характеристик)

=== CONTRACT TERMS ===
(штрафы, сроки, гарантии, требования)

=== OPTIONAL: SIMILAR TENDERS ===
(если есть данные)

=== FINAL NOTES ===
(краткое заключение эксперта)

Твоя задача — заменить ручной труд аналитика.
Отвечай строго в рамках этих правил.

=== ДОКУМЕНТ НАЧАЛО ===
{doc_text}
=== ДОКУМЕНТ КОНЕЦ ===

Вопрос пользователя:
{user_question}

Дай чёткий, структурированный ответ на русском языке.
"""
    response = model.generate_content(prompt)
    return response.text

if __name__ == "__main__":
    doc_path = "../../data/extracted/techspec_81111676.txt"


    text = load_text(doc_path)

    question = "Сделай краткое резюме технических требований и перечисли ключевые параметры."

    answer = ask_gemini_about_doc(text, question)
    print(answer)


In [ ]:
import os
print(os.getcwd())


c:\Users\esimb\Desktop\ForteBank\ai-procure\backend\app\services\LLM


In [ ]:
import os

print(os.path.abspath("ai-procure/backend/app/data/extracted/techspec_81111676.txt"))


c:\Users\esimb\Desktop\ForteBank\ai-procure\backend\app\services\LLM\ai-procure\backend\app\data\extracted\techspec_81111676.txt


In [ ]:
import os
print(os.path.exists("../../data/extracted/techspec_81111676.txt"))
